In [ ]:
# CNNs: Making the Filters Learnable
# Generated from the canonical HTML manuscript. Run this cell first.
# Source: https://github.com/Shakeri-Lab/dl-book/blob/c058d1f401fd0ead3ae59a2a8730f95489a2d9aa/chapters/part2/08-cnn.qmd

from importlib.metadata import PackageNotFoundError, version as package_version
import hashlib as _bootstrap_hashlib
import os as _bootstrap_os
from pathlib import Path as _BootstrapPath
import subprocess as _bootstrap_subprocess
import sys as _bootstrap_sys
import urllib.request as _bootstrap_urlrequest

_BOOK_REVISION = 'c058d1f401fd0ead3ae59a2a8730f95489a2d9aa'
_PINNED_REQUIREMENTS = [
    "torch==2.12.1",
    "torchvision==0.27.1",
    "numpy==2.5.1",
    "matplotlib==3.11.1"
]
_BOOK_ASSETS = [
    {
        "path": "code/dlbook/__init__.py",
        "sha256": "5f31ed4ff3aac6a557697078bfc7b9048811de3c1ccc0dc19890a6b1499ffb06"
    },
    {
        "path": "code/dlbook/supervised.py",
        "sha256": "2e035d607b3e6771bd257dd700ee271e9be6b1b1fe6a4122c62b672ef0a266ad"
    },
    {
        "path": "data/fashion-test.pt",
        "sha256": "1db79d080c51173c7df18a5e8389dd4ae20ecb0352a21be90aaa446aed612a09"
    },
    {
        "path": "data/fashion-train.pt",
        "sha256": "86a99167f14d98891de2bc34b83197727f89e178cdf9e5b019fceb7bf71cc427"
    }
]

def _installed_requirement(requirement: str) -> bool:
    name, expected = requirement.split('==', 1)
    try:
        return package_version(name) == expected
    except PackageNotFoundError:
        return False

_missing_requirements = [
    item for item in _PINNED_REQUIREMENTS if not _installed_requirement(item)
]
if _missing_requirements:
    _bootstrap_install = _bootstrap_subprocess.run(
        [_bootstrap_sys.executable, '-m', 'pip', 'install', '--quiet',
         *_missing_requirements],
        check=False, capture_output=True, text=True,
    )
    if _bootstrap_install.returncode != 0:
        raise RuntimeError(_bootstrap_install.stdout + _bootstrap_install.stderr)

_bootstrap_base = _BootstrapPath(
    _bootstrap_os.environ.get(
        'DLBOOK_NOTEBOOK_ROOT',
        '/content' if _BootstrapPath('/content').is_dir()
        else str(_BootstrapPath.home() / '.cache'),
    )
)
_BOOK_ROOT = _bootstrap_base / f'dl-book-{_BOOK_REVISION[:12]}'
_RAW_ROOT = 'https://raw.githubusercontent.com/Shakeri-Lab/dl-book/' + _BOOK_REVISION + '/'
for _record in _BOOK_ASSETS:
    _destination = _BOOK_ROOT / _record['path']
    _destination.parent.mkdir(parents=True, exist_ok=True)
    _valid = (
        _destination.is_file()
        and _bootstrap_hashlib.sha256(_destination.read_bytes()).hexdigest()
        == _record['sha256']
    )
    if not _valid:
        _temporary = _destination.with_suffix(_destination.suffix + '.part')
        _bootstrap_urlrequest.urlretrieve(_RAW_ROOT + _record['path'], _temporary)
        _digest = _bootstrap_hashlib.sha256(_temporary.read_bytes()).hexdigest()
        if _digest != _record['sha256']:
            _temporary.unlink(missing_ok=True)
            raise RuntimeError(f"Checksum mismatch for {_record['path']}")
        _temporary.replace(_destination)

(_BOOK_ROOT / 'chapters/part2').mkdir(parents=True, exist_ok=True)
_bootstrap_sys.path.insert(0, str(_BOOK_ROOT / 'code'))
_bootstrap_os.chdir(_BOOK_ROOT / 'chapters/part2')

# Hidden manuscript support required by later learner-visible cells.
# Plot-only harnesses are not exported.
import torch
from torch import nn

assert _BOOK_ROOT.is_dir()

**Plan**

1. Load imports and the Fashion-MNIST subset.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

# [1]
torch.manual_seed(6050)
development = torch.load("../../data/fashion-train.pt")
holdout = torch.load("../../data/fashion-test.pt")
X_dev = development["X"].float().unsqueeze(1) / 255.0  # (1200, 1, 28, 28)
y_dev, classes = development["y"], development["classes"]
X_test = holdout["X"].float().unsqueeze(1) / 255.0     # (600, 1, 28, 28)
y_test = holdout["y"]

split = torch.randperm(len(X_dev), generator=torch.Generator().manual_seed(6050))
fit_idx, val_idx = split[:1000], split[1000:]
X_tr, y_tr = X_dev[fit_idx], y_dev[fit_idx]      # (1000,1,28,28), (1000,)
X_val, y_val = X_dev[val_idx], y_dev[val_idx]    # (200,1,28,28), (200,)

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Recover a mystery kernel by gradient descent.

In [ ]:
# [1]
mystery = torch.tensor([[-1., 0., 1.],
                        [-2., 0., 2.],
                        [-1., 0., 1.]])            # (don't peek)
imgs = X_tr[:256]                                  # (256, 1, 28, 28)
with torch.no_grad():
    target = F.conv2d(imgs, mystery[None, None])   # what the detector does

torch.manual_seed(0)
K = 0.1 * torch.randn(1, 1, 3, 3)
K_init = K.clone().squeeze()
K.requires_grad_(True)

lr = 0.5
# [2]
for step in range(601):
    pred = F.conv2d(imgs, K)                       # predict
    loss = F.mse_loss(pred, target)                # measure
    loss.backward()
    with torch.no_grad():                          # step
        K -= lr * K.grad
        K.grad.zero_()
    if step % 200 == 0:
        print(f"step {step:3d}   loss {loss.item():.2e}")

print(f"\nlearned kernel, rounded:\n{K.detach().squeeze().round(decimals=2)}")
print(f"max |learned - mystery| = {(K.detach().squeeze() - mystery).abs().max():.3f}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Run one grayscale image through two conv layers.

In [ ]:
# [1]
conv1 = nn.Conv2d(in_channels=1, out_channels=6, kernel_size=5, padding=2)
conv2 = nn.Conv2d(in_channels=6, out_channels=16, kernel_size=5)

x = X_tr[:1]                                   # (1, 1, 28, 28): one garment
h1 = F.relu(conv1(x))
h2 = F.relu(conv2(h1))
# [2]
print(f"input   {tuple(x.shape)}")
print(f"conv1   {tuple(h1.shape)}   kernels: {tuple(conv1.weight.shape)}")
print(f"conv2   {tuple(h2.shape)}  kernels: {tuple(conv2.weight.shape)}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Run a corner detector built from two edge experts.

In [ ]:
# [1]
square = torch.zeros(28, 28)
square[7:21, 7:21] = 1.0
sobel_v = torch.tensor([[-1., 0., 1.], [-2., 0., 2.], [-1., 0., 1.]])
v = F.conv2d(square[None, None], sobel_v[None, None], padding=1).abs()
h = F.conv2d(square[None, None], sobel_v.T.contiguous()[None, None], padding=1).abs()

# [2]
experts = torch.cat([v, h], dim=1)              # (1, 2, 28, 28): two reports
corner_k = torch.full((1, 2, 5, 5), 1 / 50)     # one kernel, spanning both
corners = F.conv2d(experts, corner_k, padding=2).squeeze()

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Implement the output-size formula, three regimes.

In [ ]:
# [1]
x8 = torch.randn(1, 1, 8, 8)
# [2]
for p, s in [(0, 1), (1, 1), (1, 2)]:
    out = F.conv2d(x8, torch.randn(1, 1, 3, 3), padding=p, stride=s)
    n_out = (8 + 2 * p - 3) // s + 1
    print(f"n=8, k=3, p={p}, s={s}:  formula {n_out}  torch {tuple(out.shape[2:])}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Max and average pooling on a known grid.

In [ ]:
# [1]
t = torch.arange(16.).reshape(1, 1, 4, 4)
# [2]
print(f"input:\n{t.squeeze()}")
print(f"max pool 2x2:\n{F.max_pool2d(t, 2).squeeze()}")
print(f"avg pool 2x2:\n{F.avg_pool2d(t, 2).squeeze()}")

**Plan**

1. Prepare the inputs and fixed settings for the example.
2. Run one constructed shift that pooling cannot see.

In [ ]:
# [1]
scene = torch.zeros(1, 1, 4, 4)
scene[0, 0, 1, 0] = 9.0        # a strong clue, upper-left region
scene[0, 0, 2, 2] = 3.0        # a weaker clue, lower-right region

shifted = torch.zeros_like(scene)
shifted[0, 0, 1, 1] = 9.0      # both clues moved one pixel right
shifted[0, 0, 2, 3] = 3.0

# [2]
print(f"pooled original: {F.max_pool2d(scene,   2).squeeze().tolist()}")
print(f"pooled shifted:  {F.max_pool2d(shifted, 2).squeeze().tolist()}")

**Plan**

1. LeNet.

In [ ]:
# [1]
class LeNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(1, 6, 5, padding=2)  # (N,1,28,28)->(N,6,28,28)
        self.conv2 = nn.Conv2d(6, 16, 5)            # (N,6,14,14)->(N,16,10,10)
        self.fc1 = nn.Linear(16 * 5 * 5, 120)
        self.fc2 = nn.Linear(120, 84)
        self.fc3 = nn.Linear(84, 10)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        x = F.max_pool2d(F.relu(self.conv1(x)), 2)   # -> (N, 6, 14, 14)
        x = F.max_pool2d(F.relu(self.conv2(x)), 2)   # -> (N, 16, 5, 5)
        x = x.flatten(1)                             # -> (N, 400)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)                           # logits, (N, 10)

**Plan**

1. Define the reusable `make_mlp` helper.
2. Prepare the inputs and fixed settings for the example.
3. Parameter audit, LeNet vs. Chapter 6's MLP.

In [ ]:
# [1]
def make_mlp() -> nn.Sequential:
    return nn.Sequential(nn.Flatten(),
                         nn.Linear(784, 256), nn.ReLU(), nn.Linear(256, 10))

# [2]
lenet, mlp = LeNet(), make_mlp()
# [3]
for name, p in lenet.named_parameters():
    print(f"{name:14s} {str(tuple(p.shape)):16s} {p.numel():>7,}")
n_conv = sum(p.numel() for n, p in lenet.named_parameters() if "conv" in n)
n_lenet = sum(p.numel() for p in lenet.parameters())
n_mlp = sum(p.numel() for p in mlp.parameters())
print(f"\nLeNet: {n_lenet:,} parameters ({n_conv:,} of them convolutional)")
print(f"MLP:   {n_mlp:,} parameters")

**Plan**

1. Define the reusable helpers: `train_model` and `accuracy`.
2. Run one recipe, two architectures.
3. Report or visualize the measured result.

In [ ]:
from dlbook.supervised import fit_supervised   # Listing 4.1

# [1]
def train_model(
    model_fn, X_fit: torch.Tensor = X_tr, y_fit: torch.Tensor = y_tr,
    epochs: int = 150, batch: int = 128, seed: int = 6050
) -> nn.Module:
    # Listing 4.1 verbatim; this chapter's deltas are the budget (150
    # epochs, batch 128) and, below, the models that get trained.
    return fit_supervised(
        model_fn, X_fit, y_fit, epochs=epochs, batch=batch, seed=seed
    )

@torch.no_grad()
def accuracy(model: nn.Module, X: torch.Tensor, y: torch.Tensor) -> float:
    model.eval()
    return (model(X).argmax(1) == y).float().mean().item()

# [2]
mlp = train_model(make_mlp)
lenet = train_model(LeNet)
# [3]
for name, model in [("MLP  ", mlp), ("LeNet", lenet)]:
    print(f"{name}  train {accuracy(model, X_tr, y_tr):.1%}   "
          f"validation {accuracy(model, X_val, y_val):.1%}")

**Plan**

1. Define the reusable `shift_right` helper.
2. Prepare the inputs and fixed settings for the example.
3. Plot accuracy vs. shift, both models.

In [ ]:
# [1]
def shift_right(X: torch.Tensor, px: int) -> torch.Tensor:
    out = torch.zeros_like(X)
    if px == 0:
        return X.clone()
    out[..., px:] = X[..., :-px]
    return out

# [2]
shifts = list(range(5))
acc_mlp = [accuracy(mlp, shift_right(X_val, s), y_val) for s in shifts]
acc_cnn = [accuracy(lenet, shift_right(X_val, s), y_val) for s in shifts]
# [3]
for s in shifts:
    print(f"shift {s}px:   MLP {acc_mlp[s]:.1%}   LeNet {acc_cnn[s]:.1%}")

plt.figure(figsize=(5.5, 3.2))
plt.plot(shifts, acc_mlp, "o-", color="#E57200", lw=2, ms=7, label="MLP (Ch. 6)")
plt.plot(shifts, acc_cnn, "s-", color="#232D4B", lw=2, ms=7, label="LeNet")
plt.axhline(0.1, ls=":", color="#B8B8A8")
plt.xlabel("shift (pixels right)"); plt.ylabel("validation accuracy")
plt.ylim(0, 0.9); plt.xticks(shifts); plt.legend()
plt.tight_layout(); plt.show()

**Plan**

1. Extract first-layer kernels and their feature maps.

In [ ]:
# [1]
example_index = 3                                   # the Figure 8.3 coat
img = X_val[example_index:example_index + 1]
with torch.no_grad():
    fmaps = F.relu(lenet.conv1(img)).squeeze(0)     # (6, 28, 28)
kernels = lenet.conv1.weight.detach().squeeze(1)    # (6, 5, 5)

**Plan**

1. Run one final test-set check.
2. Report or visualize the measured result.

In [ ]:
# [1]
mlp_final = train_model(make_mlp, X_dev, y_dev)
lenet_final = train_model(LeNet, X_dev, y_dev)
# [2]
for name, model in [("MLP  ", mlp_final), ("LeNet", lenet_final)]:
    print(f"{name}  clean {accuracy(model, X_test, y_test):.1%}   "
          f"shift-2 {accuracy(model, shift_right(X_test, 2), y_test):.1%}")